In [32]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression

In [33]:
from pydvl.influence.torch import DirectInfluence
import shap

In [21]:
df = pd.read_csv('data/all_stats_renamed.csv')
print(f"Data shape: {df.shape}")
df.head()

print(df.columns.tolist())

Data shape: (196, 62)
['Player', 'pro_bowls', 'all_pros', 'mvps', 'sb_wins', 'passing_yards', 'year_start', 'year_end', 'age_start', 'age_end', 'games_started', 'completions', 'attempts', 'incompletions', 'completion_percentage', 'passing_tds', 'interceptions', 'pick_sixes', 'td_percentage', 'int_percentage', 'passer_rating', 'sacks', 'sack_yards_lost', 'sack_percentage', 'yards_per_attempt', 'adjusted_yards_per_attempt', 'adjusted_net_yards_per_attempt', 'yards_per_completion', 'yards_per_game', 'success_rate', 'wins', 'losses', 'ties', 'fourth_quarter_comebacks', 'game_winning_drives', 'playoff_passing_yards', 'playoff_games_played', 'playoff_games_started', 'playoff_completions', 'playoff_attempts', 'playoff_incompletions', 'playoff_completion_percentage', 'playoff_touchdowns', 'playoff_interceptions', 'playoff_pick_sixes', 'playoff_touchdown_percentage', 'playoff_interception_percentage', 'playoff_passer_rating', 'playoff_sacks', 'playoff_sack_yards_lost', 'playoff_sack_percentage'

## Impute Missing Values
We fill missing playoff statistics with 0, as NaN indicates the player did not play in the playoffs. For other statistics (mostly rate stats missing for older players), we use median imputation.

In [22]:
# Fill playoff stats with 0
p_cols = [c for c in df.columns if c.startswith('playoff_')]
df[p_cols] = df[p_cols].fillna(0)

# Fill other numeric missing values (e.g. Succ%, Sk, etc.) with the column median
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

print("Remaining missing values:", df.isna().sum().sum())

X = df.drop(columns=['Player', 'HOF'])
y = df['HOF']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=57, stratify=y)
print(f"Training instances: {len(X_train)}, Testing instances: {len(X_test)}")

Remaining missing values: 0
Training instances: 156, Testing instances: 40


In [23]:
sklearn_model = LogisticRegression(C=1.0, random_state=57)
sklearn_model.fit(X_train, y_train)

class TorchLogisticRegression(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)
        
    def forward(self, x):
        return self.linear(x)

n_features = X_train.shape[1]
torch_model = TorchLogisticRegression(n_features)

# Overwrite the PyTorch weights with the sklearn weights
# We use torch.no_grad() because we are modifying weights manually, not learning them
with torch.no_grad():
    # Scikit-learn stores weights as float64, PyTorch prefers float32
    weights = torch.tensor(sklearn_model.coef_, dtype=torch.float32)
    bias = torch.tensor(sklearn_model.intercept_, dtype=torch.float32)
    
    torch_model.linear.weight.copy_(weights)
    torch_model.linear.bias.copy_(bias)

/home/tbreimer/iml/nfl-hof-vis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import joblib
import shap
from pydvl.influence.torch import DirectInfluence

# ==========================================
# 0. SCALE THE DATA
# ==========================================
print("Scaling features...")
scaler = StandardScaler()

# Fit on training data ONLY to prevent data leakage, then transform both
X_train_scaled_array = scaler.fit_transform(X_train)
X_test_scaled_array = scaler.transform(X_test)

# Convert back to DataFrames to keep column names intact for SHAP
X_train_scaled = pd.DataFrame(X_train_scaled_array, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled_array, columns=X_test.columns, index=X_test.index)

# Save the scaler for future use (to unscale new data or predictions)
joblib.dump(scaler, 'hof_feature_scaler.pkl')
print("Scaler saved to 'hof_feature_scaler.pkl'")


# ==========================================
# 1. TRAIN THE SCIKIT-LEARN MODEL
# ==========================================
print("\nTraining scikit-learn Logistic Regression on scaled data...")
sklearn_model = LogisticRegression(C=1.0, random_state=57, max_iter=1000)
sklearn_model.fit(X_train_scaled, y_train)


# ==========================================
# 2. COMPUTE EXACT SHAP VALUES & FIX PLOT DATA
# ==========================================
print("Computing exact SHAP values...")
explainer = shap.LinearExplainer(sklearn_model, X_train_scaled)
shap_values = explainer(X_test_scaled)

# THE SLEIGHT OF HAND: 
# Overwrite the scaled values in the SHAP object with the original, unscaled values.
# The SHAP attributions (the importance scores) remain mathematically correct, 
# but the labels on the plot will now show the actual human-readable stats.
shap_values.data = X_test.values 

# Plot the first QB as a sanity check
first_test_qb = df.loc[X_test.index[0], 'Player']
print(f"Generating SHAP Waterfall Plot for {first_test_qb} (with unscaled labels)...")
shap.plots.waterfall(shap_values[0], show=False)


# ==========================================
# 3. BUILD PYTORCH MODEL & PERFORM TRANSPLANT
# ==========================================
print("\nPerforming Brain Transplant to PyTorch...")

class TorchLogisticRegression(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)
        
    def forward(self, x):
        return self.linear(x).squeeze(-1)

n_features = X_train_scaled.shape[1]
torch_model = TorchLogisticRegression(n_features)

with torch.no_grad():
    weights = torch.tensor(sklearn_model.coef_, dtype=torch.float32)
    bias = torch.tensor(sklearn_model.intercept_, dtype=torch.float32)
    torch_model.linear.weight.copy_(weights)
    torch_model.linear.bias.copy_(bias)


# ==========================================
# 4. PREP SCALED DATA FOR PYDVL
# ==========================================
print("Preparing Tensors...")

# Cast the SCALED data to float32 for PyTorch
X_train_t = torch.tensor(X_train_scaled.values, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
X_test_t = torch.tensor(X_test_scaled.values, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)


# ==========================================
# 5. COMPUTE THE FULL INFLUENCE MATRIX
# ==========================================
print("Computing the full influence matrix (this may take a moment)...")

loss_fn = F.binary_cross_entropy_with_logits 
influence_model = DirectInfluence(
    model=torch_model, 
    loss=loss_fn, 
    regularization=1.0 
)

influence_model.fit(train_loader)

# Calculate using the scaled tensors
influence_matrix_tensor = influence_model.influences(
    X_test_t, 
    y_test_t, 
    X_train_t, 
    y_train_t
)


# ==========================================
# 6. FORMAT & EXPORT TO CSV
# ==========================================
print("Formatting output and saving to CSV...")

influence_matrix_np = influence_matrix_tensor.numpy()

test_player_names = df.loc[X_test.index, 'Player'].values
train_player_names = df.loc[X_train.index, 'Player'].values

row_labels = [f"Test: {name}" for name in test_player_names]
col_labels = [f"Train: {name}" for name in train_player_names]

influence_df = pd.DataFrame(
    data=influence_matrix_np,
    index=row_labels,
    columns=col_labels
)

csv_filename = "full_influence_matrix.csv"
influence_df.to_csv(csv_filename)

print(f"\nSuccess! Full influence matrix exported to: {csv_filename}")

Scaling features...
Scaler saved to 'hof_feature_scaler.pkl'

Training scikit-learn Logistic Regression on scaled data...
Computing exact SHAP values...
Generating SHAP Waterfall Plot for Chad Pennington (with unscaled labels)...

Performing Brain Transplant to PyTorch...
Preparing Tensors...
Computing the full influence matrix (this may take a moment)...
Formatting output and saving to CSV...

Success! Full influence matrix exported to: full_influence_matrix.csv
